# Transformation 


In [0]:
%sql
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_key,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.creation_date AS create_date
FROM workspace.silver.crm_cust ci
LEFT JOIN workspace.silver.erp_cust ca
    ON ci.customer_key = ca.customer_key
LEFT JOIN workspace.silver.erp_loc la
    ON ci.customer_key = la.customer_key

In [0]:
## Simple way - give the above query as df statement to load as DataFrame
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_number,
    ci.customer_id,
    ci.customer_key,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.creation_date AS create_date
FROM workspace.silver.crm_cust ci
LEFT JOIN workspace.silver.erp_cust ca
    ON ci.customer_key = ca.customer_key
LEFT JOIN workspace.silver.erp_loc la
    ON ci.customer_key = la.customer_key
"""

In [0]:
df = spark.sql(query)

In [0]:
display(df.limit(10))

#Validating why birthdate has null values

In [0]:
%sql
select * from workspace.silver.erp_loc Limit 5;

In [0]:
%sql
select * from workspace.silver.erp_cust e,workspace.silver.crm_cust c where c.customer_key = e.customer_id;

In [0]:
%sql
-- Check how many customer_keys from crm_cust exist in erp_cust
SELECT 
    'crm_cust' as source_table,
    COUNT(DISTINCT customer_key) as distinct_customer_keys,
    MIN(customer_key) as sample_min,
    MAX(customer_key) as sample_max
FROM workspace.silver.crm_cust

UNION ALL

SELECT 
    'erp_cust' as source_table,
    COUNT(DISTINCT customer_key) as distinct_customer_keys,
    MIN(customer_key) as sample_min,
    MAX(customer_key) as sample_max
FROM workspace.silver.erp_cust;

In [0]:
%sql
-- Sample from both tables to understand the relationship
SELECT 'crm_cust' as source, customer_id, customer_key, first_name, last_name
FROM workspace.silver.crm_cust
LIMIT 5;

In [0]:
%sql
-- Check if any crm_cust keys exist in erp_cust
SELECT 
    COUNT(*) as matching_keys
FROM workspace.silver.crm_cust ci
INNER JOIN workspace.silver.erp_cust ca
    ON ci.customer_key = ca.customer_key;

In [0]:
%sql
-- Find which customer_keys actually match between the tables
SELECT 
    ci.customer_id,
    ci.customer_key,
    ci.first_name,
    ca.birth_date,
    ca.gender as erp_gender
FROM workspace.silver.crm_cust ci
INNER JOIN workspace.silver.erp_cust ca
    ON ci.customer_key = ca.customer_key
LIMIT 10;

# Write to Gold Layer

     

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

     

In [0]:
%sql
select * from workspace.gold.dim_customers LIMIT 10;